# Assignment 2A — Part B: Structured & Iterative Reasoning  [8 Marks]

Reasoning-augmented prompting applied to the same five Part A queries plus two additional
multi-step domain problems.

| Step | Deliverable | Marks |
|---|---|---|
| B1 | CoT, ToT (+ Step-Back) × 2 domain problems + comparison table | 4 |
| B2 | PAL (code-based, executed) + ReACT (≥3 cycles) + Self-Reflection | 4 |


> **Backend switch.** Every cell runs with `BACKEND = "offline"`, a deterministic
> corpus-grounded simulator, so the tables, the chain comparison, the PAL execution and the
> preference dataset can be produced and checked on CPU with no model download. For the marked
> run, open a GPU runtime, set `BACKEND = "adapter"`, and re-run; the code path is identical and
> every saved artefact records which backend produced it.


In [ ]:
# Run once in Colab:
# !pip install -q -r assignment2a/requirements_colab.txt


In [ ]:
from pathlib import Path
import sys, json

A2 = Path.cwd()
if (A2 / "assignment2a").exists():
    A2 = A2 / "assignment2a"
sys.path.insert(0, str(A2 / "src"))

import a2_paths
from a2_paths import DEFAULT_SFT_ADAPTER, base_model_source, sft_adapter_path
from domain_lm import DomainLM, wrap

# "offline" = deterministic CPU simulator for local validation.
# "adapter" = real generation from the Assignment 1 QLoRA adapter (needs a GPU runtime).
BACKEND = "offline"
SFT_ADAPTER = DEFAULT_SFT_ADAPTER

ARTIFACTS = A2 / "outputs"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

lm = DomainLM(
    base_model_id=base_model_source(),
    adapter_path=sft_adapter_path(SFT_ADAPTER) if BACKEND == "adapter" else None,
    backend=BACKEND,
    label=f"sft_{SFT_ADAPTER}",
)
print(json.dumps(lm.describe(), indent=2))


In [ ]:
import pandas as pd
import reasoning_strategies as rs
from assignment2_queries import DOMAIN_QUERIES, PAL_PROBLEMS, REACT_QUESTION, REASONING_PROBLEMS

for p in REASONING_PROBLEMS:
    print(p["id"], "->", wrap(p["question"]), "\n")


## B1 — Chain-of-Thought

`Let's think step by step.` is appended to the instruction and the model is asked to number each
reasoning step and finish with an `Answer:` line. The same query is also run with a plain prompt so
the two can be compared directly.

**When to use it in this domain:** any multi-step clinical query where the conclusion depends on
chaining a definition to a limiting condition — dose selection, test interpretation, risk weighing.


In [ ]:
cot_rows = [rs.run_cot(lm, p["question"]) for p in REASONING_PROBLEMS]
cot_rows += [rs.run_cot(lm, q) for q in DOMAIN_QUERIES]

display(pd.DataFrame([{
    "problem": (REASONING_PROBLEMS + [{"id": f"Q{i}"} for i in range(1, 6)])[i]["id"],
    "standard": r["standard_score"]["total"],
    "cot": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["standard_score"]["total"], 2),
    "steps": r["reasoning_steps"],
} for i, r in enumerate(cot_rows)]))


In [ ]:
r = cot_rows[0]
print("STANDARD PROMPT\n", wrap(r["standard_answer"]))
print("\n" + "=" * 100)
print("CHAIN-OF-THOUGHT\n", wrap(r["answer"]))
print("\nExtracted final answer:\n", wrap(r["final_answer"]))


## B1 — Tree-of-Thought

Three candidate reasoning paths are generated at `temperature=0.7`, each forced down a different
lens (*mechanism*, *measurement*, *risk-and-monitoring*). Every path is scored with the shared
rubric and the best-scoring path is selected.

**When to use it in this domain:** a complex problem with genuine alternatives, e.g. deciding how a
test should be deployed in two different prevalence settings, where the mechanism view and the
risk view lead to different emphases.


In [ ]:
tot_rows = [rs.run_tot(lm, p["question"]) for p in REASONING_PROBLEMS]

for problem, t in zip(REASONING_PROBLEMS, tot_rows):
    print("=" * 100)
    print(problem["id"], "| path scores:", t["path_scores"], "| selected:", t["selected_lens"])
    for p in t["paths"]:
        print(f"\n-- path {p['path']} ({p['lens']}) score={p['score']['total']}")
        print(wrap(p["text"][:600]))


## B1 — Step-Back prompting

The model first abstracts the specific question into a general-principle question, answers that,
then applies the principle back to the specific question. This is useful when a specific clinical
question is really an instance of a general measurement or evidence principle.


In [ ]:
sb_rows = [rs.run_stepback(lm, p["question"]) for p in REASONING_PROBLEMS]
for problem, s in zip(REASONING_PROBLEMS, sb_rows):
    print("=" * 100)
    print(problem["id"])
    print("step-back question:", wrap(s["stepback_question"]))
    print("principle:", wrap(s["principle"]))
    print("final:\n", wrap(s["answer"]))


In [ ]:
# B1 comparison table across techniques, on the two multi-step problems.
b1_table = []
for i, problem in enumerate(REASONING_PROBLEMS):
    b1_table.append({
        "problem": problem["id"],
        "standard": cot_rows[i]["standard_score"]["total"],
        "cot": cot_rows[i]["score"]["total"],
        "tot_best": tot_rows[i]["score"]["total"],
        "step_back": sb_rows[i]["score"]["total"],
        "tot_paths": tot_rows[i]["path_scores"],
    })
display(pd.DataFrame(b1_table))


### B1 write-up

- **CoT vs standard:** report the `delta` column. CoT helps most where the standard answer stopped
  at a definition; it helps least where the query was already single-hop.
- **ToT vs CoT:** ToT is CoT run three times with a forced change of lens plus an explicit scoring
  step. It wins when the paths genuinely diverge (look at the spread in `tot_paths`); when the
  spread is small, ToT is paying 3× the generation cost for nothing.
- **Step-Back:** strongest where the specific question is an instance of a general principle,
  because the principle supplies the limiting condition the direct answer omitted.


## B2 — PAL (Program-Aided Language model)

For calculation-type clinical questions, prose arithmetic is unreliable. PAL makes the model emit
Python, and the **executed** value is the answer.

**Security note.** Model output is untrusted input, so the generated program is not executed
directly. `reasoning_strategies.safe_execute` parses it to an AST first and rejects any import
other than `math`, any dunder or non-`math` attribute access, loops, `with`/`try`/`raise`, lambdas,
class definitions, and the `eval`/`exec`/`open`/`__import__` name family; it then runs the code with
a reduced builtins allow-list. A failure is fed back to the model as a repair turn.


In [ ]:
pal_rows = [rs.run_pal(lm, p) for p in PAL_PROBLEMS]
for problem, r in zip(PAL_PROBLEMS, pal_rows):
    print("=" * 100)
    print(problem["id"], "|", wrap(problem["question"]))
    print("\nGenerated program:\n" + r["code"])
    print(f"\nExecuted result: {r['result']} {r.get('unit', '')}"
          f" | expected {problem['expected']} | match={r['matches_expected']}")


In [ ]:
display(pd.DataFrame([{
    "id": r["problem_id"], "result": r["result"], "unit": r.get("unit"),
    "expected": r["expected"], "matches": r["matches_expected"],
    "attempts": len(r["attempts"]),
} for r in pal_rows]))


In [ ]:
# The sandbox must reject anything that reaches outside arithmetic.
for bad in ["import os\nresult = os.getcwd()",
            "result = __import__('os').listdir('.')",
            "result = open('secrets.txt').read()",
            "while True:\n    pass",
            "result = (1).__class__.__bases__"]:
    try:
        rs.safe_execute(bad)
        print("ACCEPTED (unexpected!):", bad)
    except Exception as exc:
        print(f"rejected [{type(exc).__name__}]: {exc}")


## B2 — ReACT (Thought → Action → Observation)

The model may call `search_corpus[...]`, `lookup_topic[...]`, `calculate[...]`, or `finish[...]`.
Each action is executed against the Assignment 1 corpus and the real observation is appended to the
scratchpad before the next turn, so the model reasons over retrieved evidence rather than memory.


In [ ]:
react = rs.run_react(lm, REACT_QUESTION)
print("QUESTION:\n", wrap(REACT_QUESTION))
print(f"\ncycles: {react['cycles']} (assignment minimum is 3)\n")
for step in react["trace"]:
    print("-" * 100)
    print(f"[cycle {step['cycle']}]")
    print("Thought:     ", wrap(step["thought"]))
    print("Action:      ", f"{step['action']}[{step.get('argument', '')}]")
    print("Observation: ", wrap(step["observation"][:500]))
print("=" * 100)
print("FINAL ANSWER:\n", wrap(react["answer"]))


## B2 — Self-Reflection

A draft answer is produced, the model critiques its own draft for missing corpus facts, missing
limiting conditions and missing monitoring implications, then revises. The revised answer is one of
the `chosen` sources in the Part C preference dataset.


In [ ]:
reflect_rows = [rs.run_self_reflection(lm, p["question"]) for p in REASONING_PROBLEMS]
reflect_rows += [rs.run_self_reflection(lm, q) for q in DOMAIN_QUERIES]

display(pd.DataFrame([{
    "draft_score": r["draft_score"]["total"],
    "revised_score": r["score"]["total"],
    "delta": round(r["score"]["total"] - r["draft_score"]["total"], 2),
} for r in reflect_rows]))

print(wrap(reflect_rows[0]["critique_and_revision"]))


In [ ]:
artefact = {
    "backend": lm.describe(),
    "b1_cot": cot_rows,
    "b1_tot": tot_rows,
    "b1_step_back": sb_rows,
    "b1_table": b1_table,
    "b2_pal": pal_rows,
    "b2_react": react,
    "b2_self_reflection": reflect_rows,
}
path = ARTIFACTS / "part_b_results.json"
path.write_text(json.dumps(artefact, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("saved", path)


## Part B conclusion

| Technique | Cost (generations) | Best use in this domain | Main limitation |
|---|---:|---|---|
| CoT | 1 | Any multi-step clinical query | Reasoning can be fluent but wrong; no external check |
| ToT | 3 + scoring | Problems with genuinely competing framings | 3× cost; only pays off when paths diverge |
| Step-Back | 3 | Specific question that is an instance of a principle | Can drift to the general and under-answer the specific |
| PAL | 1–3 | Dose, clearance, NNT, PPV — anything numeric | Needs a sandbox; wrong formula still runs cleanly |
| ReACT | 1 per cycle | Questions needing corpus lookup | Loops can stall; needs a cycle cap |
| Self-Reflection | 2 | Polishing a draft that missed caveats | Can pad length without adding facts |

PAL is the only technique here that produces a *verifiable* answer, because the arithmetic is
executed rather than asserted. ReACT is the only one that grounds each step in retrieved evidence.
Those two properties are what the Part C preference dataset is trying to transfer into the weights.
